In [ ]:
# O Filtro de Letalidade (Vitality Screening)
import pandas as pd
from Bio import SeqIO

# --- CONFIGURAÇÃO ---
ARQUIVO_CSV = "dados_biosseguranca.csv"   # Seu arquivo gerado no passo anterior
ARQUIVO_FASTA = "../c_abscissum_data/ncbi_dataset/data/GCA_023376855.1/cds_from_genomic.fna" # O FASTA original

# Palavras-chave de genes ESSENCIAIS (Baseado em literatura de RNAi antifúngico)
# Estes termos indicam processos celulares vitais que, se pararem, matam o fungo.
KEYWORDS_VITAL = [
    "ribosomal", "ribosome",          # Síntese proteica (Morte rápida)
    "proteasome", "ubiquitin",        # Limpeza celular
    "atp synthase", "cytochrome",     # Respiração celular (Energia)
    "polymerase",                     # Replicação de DNA/RNA
    "translation initiation", "elongation factor", # Tradução
    "chitin synthase", "glucan synthase", # Parede celular (Explosão osmótica)
    "tubulin", "actin", "kinesin",    # Citoesqueleto (mas cuidado com off-targets aqui)
    "scytalone", "polyketide",        # Síntese de melanina (Para infecção)
    "heat shock"                      # Resposta a estresse
]

# Termos para EXCLUIR imediatamente
KEYWORDS_IGNORE = [
    "hypothetical", "uncharacterized", "predicted", # Tiro no escuro
    "putative protein", "domain-containing"         # Pouca especificidade
]

# --- EXECUÇÃO ---
print("--- INICIANDO FILTRAGEM POR LETALIDADE ---")

# 1. Carregar a lista de aprovados
df = pd.read_csv(ARQUIVO_CSV, sep=";")
genes_aprovados_ids = set(df[df['Global_Status'] == 'APPROVED']['GeneID'])
print(f"Genes incialmente aprovados (Segurança): {len(genes_aprovados_ids)}")

# 2. Carregar anotações do FASTA e filtrar
candidatos_elite = []

print("Minerando descrições funcionais...")

for record in SeqIO.parse(ARQUIVO_FASTA, "fasta"):
    if record.id in genes_aprovados_ids:
        desc = record.description.lower()
        
        # Regra 1: Não pode ser hipotético
        if any(bad in desc for bad in KEYWORDS_IGNORE):
            continue
            
        # Regra 2: Tem que ter função vital
        if any(good in desc for good in KEYWORDS_VITAL):
            # Salva tupla (ID, Descrição)
            candidatos_elite.append( (record.id, record.description) )

# --- RELATÓRIO ---
print("\n" + "="*60)
print(f"RESULTADO DO FUNIL DE EFICÁCIA")
print("="*60)
print(f"Total Inicial (Seguros): {len(genes_aprovados_ids)}")
print(f"Total Eliminado (Hipotéticos/Não-Vitais): {len(genes_aprovados_ids) - len(candidatos_elite)}")
print(f"CANDIDATOS DE ELITE (Seguros + Vitais): {len(candidatos_elite)}")
print("="*60)

# Salva a lista de elite
with open("candidatos_elite.txt", "w") as f:
    for gid, desc in candidatos_elite:
        f.write(f"{gid}\t{desc}\n")
        
print(f"\nLista de elite salva em 'candidatos_elite.txt'")

--- INICIANDO FILTRAGEM POR LETALIDADE ---
Genes incialmente aprovados (Segurança): 12503
Minerando descrições funcionais...

RESULTADO DO FUNIL DE EFICÁCIA
Total Inicial (Seguros): 12503
Total Eliminado (Hipotéticos/Não-Vitais): 12081
CANDIDATOS DE ELITE (Seguros + Vitais): 422

Lista de elite salva em 'candidatos_elite.txt'
